# urbanev on isleme

bolge bazinda bir saat sonraki doluluk orani icin tablo hazirliyorum
ozellikler yalniz gecmisi gorur fiyat bir saat kaydirilir
bolme kronolojik egitim eylul-aralik 2022 dogrulama ocak 2023 test subat 2023
test satirlarina sadece sayi icin bakiyorum

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.urbanev import PRICE_COLUMNS, build_zone_table, load_zone_tables, occupancy_ratio

tables = load_zone_tables(project_dir / "data" / "raw" / "UrbanEVDataset")
ratio = occupancy_ratio(tables["occupancy"], tables["zones"])
print("once:", tables["occupancy"].shape, "hucre:", tables["occupancy"].size, "eksik:", int(tables["occupancy"].isna().sum().sum()))
print("1 uzeri oran:", int(ratio.gt(1).sum().sum()))

once: (4344, 275) hucre: 1194600 eksik: 0
1 uzeri oran: 0


## model tablosu

ilk 168 saat onceki hafta bilgisi olmadigi icin cikiyor

In [2]:
table = build_zone_table(ratio, {name: tables[name] for name in PRICE_COLUMNS})
print("sonra:", table.shape, "saat:", table["hour"].nunique(), "bolge:", table["zone"].nunique())
print("eksik:", int(table.isna().sum().sum()))
print(list(table.columns))
table.head(3)

sonra: (1148400, 15) saat: 4176 bolge: 275


eksik: 0
['hour', 'zone', 'occupancy_ratio', 'prev_hour', 'prev_day', 'prev_week', 'mean_6h', 'mean_24h', 'e_price', 's_price', 'hour_local', 'day_of_week', 'is_weekend', 'is_holiday', 'split']


,hour,zone,occupancy_ratio,prev_hour,prev_day,prev_week,mean_6h,mean_24h,e_price,s_price,hour_local,day_of_week,is_weekend,is_holiday,split
0,2022-09-08,102,0.633333,0.633333,0.633333,0.566667,0.633333,0.633333,0.924000,0.746,0,3,0,0,train
1,2022-09-08,104,0.052632,0.052632,0.052632,0.052632,0.052632,0.035088,0.923333,0.760,0,3,0,0,train
2,2022-09-08,105,0.139785,0.150538,0.172043,0.161290,0.123656,0.138889,1.124167,0.750,0,3,0,0,train


## bolme ozeti

In [3]:
summary = table.groupby("split").agg(
    satir=("zone", "size"),
    saat=("hour", "nunique"),
    ilk=("hour", "min"),
    son=("hour", "max"),
    ortalama=("occupancy_ratio", "mean"),
    std=("occupancy_ratio", "std"),
).loc[["train", "val", "test"]]
summary["pay_yuzde"] = (summary["satir"] / len(table) * 100).round(1)
summary.round(4)

/var/folders/8z/0rdc7bl52jx880m6jz3rvv0r0000gn/T/ipykernel_97466/519943382.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  summary.round(4)


,satir,saat,ilk,son,ortalama,std,pay_yuzde
split,,,,,,,
train,759000,2760,2022-09-08,2022-12-31 23:00:00,0.2306,0.1674,66.1
val,204600,744,2023-01-01,2023-01-31 23:00:00,0.2181,0.1657,17.8
test,184800,672,2023-02-01,2023-02-28 23:00:00,0.2233,0.1702,16.1


## olcekleme

fiyatlarda ortalama ve standart sapmayi yalniz egitimden ogreniyorum
burada degerleri kaydediyorum modeli egitirken ayni degerler uygulanacak

In [4]:
train = table[table["split"] == "train"]
scaler = train[list(PRICE_COLUMNS)].agg(["mean", "std"]).round(4)
print(scaler)
out = project_dir / "data" / "processed" / "urbanev_zone_hourly.csv.gz"
table.to_csv(out, index=False)
print(out.name, "yazildi")

      e_price  s_price
mean   0.9517   0.7229
std    0.1946   0.1288


urbanev_zone_hourly.csv.gz yazildi


## sonuc

tablo modele hazir egitim dogrulama test ayrildi
hava durumu dosyasi indirdigimiz surumde olmadigi icin bu veri setinde kullanilmadi